In [2]:
!pip install pandas

In [3]:
from pathlib import Path

import pandas as pd

In [18]:
from dataclasses import dataclass

@dataclass
class TestResult:
    time_res: int
    test_name: str

def parse_file(file: Path) -> list[TestResult]:
    results: list[TestResult] = list()
    with open(file) as f:
        for line in f.readlines():
            tokens = list(line.split())
            if len(tokens) > 1 and tokens[1] in ["'single_search'", "tests", "'freqs_search'"]:
                time_data = float(tokens[-2])
                time_type = tokens[-1]
                if time_type == "milliseconds":
                    time_data *= 1000
                if time_type == "seconds":
                    time_data *= 1000 * 1000
                results.append(TestResult(int(time_data), tokens[1]))
    return results


In [21]:
root_dir = Path("bench_res")
df = pd.DataFrame(columns=['имя решения', 'имя теста', 'имя сабтеста', 'время'])
for test in root_dir.iterdir():
    if test.is_dir():
        #if test.name not in ["main", "2_term_with_check_before_lead", "rec_search_tree", "2_term"]:
        #    continue
        if test.name not in ["main","1_term"]:
            continue
        for subtest in test.iterdir():

            results = parse_file(subtest)
            for val in results:
                row = pd.DataFrame([{
                    "имя решения": test.name,
                    "имя теста": subtest.name,
                    "имя сабтеста": val.test_name,
                    "время": val.time_res
                }])
                df = pd.concat([df, row], ignore_index=True)

        

In [23]:
list_of_dicts = df.to_dict('records')

diffs = dict()
for val in list_of_dicts:
    if val["имя решения"] == "main":
        if val["имя теста"] not in diffs:
            diffs[val["имя теста"]] = dict()
        diffs[val["имя теста"]][val["имя сабтеста"]] = val["время"]
    else:
        diffs[val["имя теста"]][val["имя сабтеста"]] = val["время"] - diffs[val["имя теста"]][val["имя сабтеста"]]
for key, store in diffs.items():
    for subkey, diff in store.items():
        print(key, subkey, diff)

interval_bench_big_data_freqs_discrete.json__mmap___1_5simd_results.txt 'single_search' 2
interval_bench_big_data_freqs_discrete.json__mmap___1_5simd_results.txt 'freqs_search' -135
interval_bench_big_data_freqs_discrete.json__mmap___1_5simd_results.txt tests -135
interval_bench_big_data_freqs_discrete.json__fs___1_5simd_results.txt 'single_search' 29
interval_bench_big_data_freqs_discrete.json__fs___1_5simd_results.txt 'freqs_search' -169
interval_bench_big_data_freqs_discrete.json__fs___1_5simd_results.txt tests -136
interval_bench_repeat.json__mmap___1_5simd_results.txt 'single_search' 3284
interval_bench_repeat.json__mmap___1_5simd_results.txt 'freqs_search' 15687
interval_bench_repeat.json__mmap___1_5simd_results.txt tests 18976
interval_bench_freqs_equal.json__mmap___1_5simd_results.txt 'single_search' 1221
interval_bench_freqs_equal.json__mmap___1_5simd_results.txt 'freqs_search' 1294
interval_bench_freqs_equal.json__mmap___1_5simd_results.txt tests 2512
interval_bench_big_data_

In [7]:
df_filtered = df[df['имя теста'] == 'interval_bench_big_data_freqs_discrete.json__mmap___1_5simd_results.txt']
print(df_filtered)

   имя решения                                          имя теста  \
0         main  interval_bench_big_data_freqs_discrete.json__m...   
1         main  interval_bench_big_data_freqs_discrete.json__m...   
2         main  interval_bench_big_data_freqs_discrete.json__m...   
54      1_term  interval_bench_big_data_freqs_discrete.json__m...   
55      1_term  interval_bench_big_data_freqs_discrete.json__m...   
56      1_term  interval_bench_big_data_freqs_discrete.json__m...   

       имя сабтеста время  
0   'single_search'   244  
1    'freqs_search'  1042  
2             tests  1324  
54  'single_search'   246  
55   'freqs_search'   907  
56            tests  1189  


In [8]:
#df.to_csv('output.csv', index=False)
